# Industry Opportunity Component

### 0. Overview

This notebook constructs the industry opportunity component of the Opportunity Layer. The component uses the industry attribute retained in each representative demographic profile to refine the destination category of commuting trips.

### 1. Import Libraries

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

### 2. Set Up File Paths

In [2]:
project_root = Path.cwd().parents[1]

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

data_path = project_root / "data"
output_path = project_root / "output"

representative_profile_space = (
    output_path
    / "representative_profiles"
    / "constrained_representative_profile_space.parquet"
)

industry_output_dir = (
    output_path
    / "opportunity_layer"
)

industry_output_path = (
    industry_output_dir
    / "industry_to_poi_mapping.json"
)

### 3. Load Industry Categories

The industry attribute is loaded from the constrained representative demographic profiles.

In [3]:
industry_data = pd.read_parquet(
    representative_profile_space,
    columns=["Industry"]
)

industry_data.head()

,Industry
0,Does not apply
1,Does not apply
2,Does not apply
3,Does not apply
4,Does not apply


### 4. Data Inspection

In [4]:
print(f"Rows loaded: {len(industry_data):,}")
print(
    f"Missing Industry values: "
    f"{industry_data['Industry'].isna().sum():,}"
)
print(
    f"Unique Industry categories: "
    f"{industry_data['Industry'].nunique()}"
)

Rows loaded: 2,147,608
Missing Industry values: 0
Unique Industry categories: 19


In [5]:
industry_categories = sorted(
    industry_data["Industry"].unique().tolist())

display(industry_categories)

['A Agriculture, forestry and fishing',
 'B Mining and quarrying',
 'C Manufacturing',
 'D Electricity, gas, steam and air conditioning supply',
 'Does not apply',
 'E Water supply; sewerage, waste management and remediation activities',
 'F Construction',
 'G Wholesale and retail trade; repair of motor vehicles and motor cycles',
 'H Transport and storage',
 'I Accommodation and food service activities',
 'J Information and communication',
 'K Financial and insurance activities',
 'L Real estate activities',
 'M Professional, scientific and technical activities',
 'N Administrative and support service activities',
 'O Public administration and defence; compulsory social security',
 'P Education',
 'Q Human health and social work activities',
 'R, S, T, U Other']

### 5. Industry to POI Category Mapping

A mapping was created between the 19 industry categories retained in the demographic layer and the framework POI categories. Industries with a clear semantic correspondence were assigned a specific POI category, while broader or ambiguous industries were mapped to Employment. Does not apply was assigned None because it does not trigger industry-specific refinement.

In [6]:
import sys

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [7]:
from ontology.INDUSTRY_TO_POI_MAPPING import INDUSTRY_TO_POI_MAPPING

In [8]:
unmapped_industries = (
    set(industry_categories)
    - set(INDUSTRY_TO_POI_MAPPING)
)

unexpected_industries = (
    set(INDUSTRY_TO_POI_MAPPING)
    - set(industry_categories)
)

print(f"Unmapped industries: {unmapped_industries}")
print(f"Unexpected industries: {unexpected_industries}")

Unmapped industries: set()
Unexpected industries: set()


In [9]:
industry_mapping_table = pd.DataFrame(
    INDUSTRY_TO_POI_MAPPING.items(),
    columns=["Industry", "POI Category"]
)

display(industry_mapping_table)

,Industry,POI Category
0,"A Agriculture, forestry and fishing",Employment
1,B Mining and quarrying,Employment
2,C Manufacturing,Employment
3,"D Electricity, gas, steam and air conditioning...",Employment
4,Does not apply,None
5,"E Water supply; sewerage, waste management and...",Employment
6,F Construction,Employment
7,G Wholesale and retail trade; repair of motor ...,Retail
8,H Transport and storage,Transport
9,I Accommodation and food service activities,"[Accommodation, Food and Beverage]"


### 7. Opportunity Probability Calculation

Most industries map to a single POI category and therefore receive a Probability of `1.0`. `I Accommodation and food service activities` maps to both `Accommodation` and `Food and Beverage`, so their Probabilitys are calculated from POI availability using the same log-transformation method applied in the Opportunity Layer.


In [10]:
project_root = Path.cwd().parents[1]
input_file_path = project_root / "data" / "POI" / "processed" / "manchester" / "District_manchester_poi_counts.xlsx"

In [11]:
poi_counts = pd.read_excel(input_file_path)

In [12]:
poi_counts

,POI Category,POI Count
0,Accommodation,139
1,Education,618
2,Employment,6031
3,Food and Beverage,2731
4,Healthcare,942
5,Personal Services,1613
6,Public Services,2899
7,Recreation,1256
8,Retail,2856
9,Transport,3401


In [13]:
import numpy as np

poi_counts_processed = poi_counts.copy()

def log_transform_poi_counts(poi_counts_df):
    """
    Apply log(1 + x) transformation to POI counts.
    """

    df = poi_counts_df.copy()

    df["Log Count"] = np.log1p(df["POI Count"])

    return df


poi_counts_processed = log_transform_poi_counts(poi_counts)

display(poi_counts_processed)

,POI Category,POI Count,Log Count
0,Accommodation,139,4.941642
1,Education,618,6.428105
2,Employment,6031,8.704834
3,Food and Beverage,2731,7.912789
4,Healthcare,942,6.849066
5,Personal Services,1613,7.386471
6,Public Services,2899,7.972466
7,Recreation,1256,7.136483
8,Retail,2856,7.957527
9,Transport,3401,8.132119


In [14]:
industry_i_Probabilitys = poi_counts_processed[
    poi_counts_processed["POI Category"].isin([
        "Accommodation",
        "Food and Beverage"
    ])
].copy()

industry_i_Probabilitys["POI Category Probability"] = (
    industry_i_Probabilitys["Log Count"]
    / industry_i_Probabilitys["Log Count"].sum()
)

display(industry_i_Probabilitys)

,POI Category,POI Count,Log Count,POI Category Probability
0,Accommodation,139,4.941642,0.384431
3,Food and Beverage,2731,7.912789,0.615569


In [15]:
industry_i_distribution = dict(
    zip(
        industry_i_Probabilitys["POI Category"],
        industry_i_Probabilitys["POI Category Probability"].astype(float)
    )
)

industry_i_distribution

{'Accommodation': 0.3844310320157171, 'Food and Beverage': 0.6155689679842828}

### 8. Finish Opportunity Layer Industry Component

In [16]:
industry_to_poi_distribution = {}

for industry, target in INDUSTRY_TO_POI_MAPPING.items():

    if target is None:
        industry_to_poi_distribution[industry] = None

    elif isinstance(target, list):
        selected_pois = poi_counts_processed[
            poi_counts_processed["POI Category"].isin(target)
        ].copy()

        selected_pois["POI Category Probability"] = (
            selected_pois["Log Count"]
            / selected_pois["Log Count"].sum()
        )

        industry_to_poi_distribution[industry] = dict(
            zip(
                selected_pois["POI Category"],
                selected_pois["POI Category Probability"].astype(float)
            )
        )

    else:
        industry_to_poi_distribution[industry] = {
            target: 1.0
        }

display(industry_to_poi_distribution)

{'A Agriculture, forestry and fishing': {'Employment': 1.0},
 'B Mining and quarrying': {'Employment': 1.0},
 'C Manufacturing': {'Employment': 1.0},
 'D Electricity, gas, steam and air conditioning supply': {'Employment': 1.0},
 'Does not apply': None,
 'E Water supply; sewerage, waste management and remediation activities': {'Employment': 1.0},
 'F Construction': {'Employment': 1.0},
 'G Wholesale and retail trade; repair of motor vehicles and motor cycles': {'Retail': 1.0},
 'H Transport and storage': {'Transport': 1.0},
 'I Accommodation and food service activities': {'Accommodation': 0.3844310320157171,
  'Food and Beverage': 0.6155689679842828},
 'J Information and communication': {'Employment': 1.0},
 'K Financial and insurance activities': {'Employment': 1.0},
 'L Real estate activities': {'Employment': 1.0},
 'M Professional, scientific and technical activities': {'Employment': 1.0},
 'N Administrative and support service activities': {'Employment': 1.0},
 'O Public administra

### 9. Validation

In [17]:
# AI-assisted implementation using ChatGPT
# See AI_PROMPTS.md

for industry, distribution in industry_to_poi_distribution.items():
    if distribution is not None:
        assert np.isclose(
            sum(distribution.values()),
            1.0,
            atol=1e-8
        ), f"{industry} POI distribution does not sum to one."

print("PASS: All industry POI distributions sum to one.")

PASS: All industry POI distributions sum to one.


### 10. Export Industry Component as JSON File

In [18]:
import json

output_file_path = (
    project_root
    / "output"
    / "opportunity_layer"
    / "industry_to_poi_mapping.json"
)

output_file_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(output_file_path, "w", encoding="utf-8") as file:
    json.dump(
        industry_to_poi_distribution,
        file,
        indent=4,
        ensure_ascii=False
    )

print(f"Exported to: {output_file_path}")

Exported to: c:\Users\user\OneDrive - University of Stirling\Desktop\dissertation\mobility-modelling-framework\output\opportunity_layer\industry_to_poi_mapping.json
